In [1]:
import sys
import importlib

# Remove all cached versions of our modules
mods_to_remove = [key for key in sys.modules 
                  if any(key.startswith(x) 
                  for x in ["configs", "model", "dataset", 
                             "training", "losses", "utils"])]

for mod in mods_to_remove:
    del sys.modules[mod]
    print(f"Removed: {mod}")

print("\nCache cleared!")


Cache cleared!


In [5]:
import sys
sys.path.insert(0, r"D:\Projects\ODM Project")

import configs.config as cfg
import importlib
importlib.reload(cfg)

print(dir(cfg))   # should show DEVICE in the list
print(f"\nDEVICE = {cfg.DEVICE}")

['BACKBONE', 'BATCH_SIZE', 'CHECKPOINT_DIR', 'COCO_ROOT', 'CROWD_DENSITY_HIGH', 'CROWD_DENSITY_MED', 'DENSITY_GRID_SIZE', 'DEVICE', 'HEATMAP_ALPHA', 'HEATMAP_SIGMA', 'LOG_DIR', 'LR', 'LR_GAMMA', 'LR_MILESTONES', 'LR_SCHEDULE', 'LR_STEP_SIZE', 'MAX_DETECTIONS', 'MOMENTUM', 'NMS_IOU_THRESH', 'NUM_CLASSES', 'NUM_EPOCHS', 'NUM_WORKERS', 'OUTPUT_DIR', 'PRETRAINED', 'SCORE_THRESHOLD', 'TARGET_CLASSES', 'TRAIN_ANN', 'TRAIN_IMG_DIR', 'VAL_ANN', 'VAL_IMG_DIR', 'WARMUP_EPOCHS', 'WEIGHT_DECAY', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'torch']

DEVICE = cuda


In [6]:
import sys
import torch
from torch.cuda.amp import GradScaler
sys.path.insert(0, r"D:\Projects\ODM Project")

import configs.config as cfg
from model.detector       import build_detector
from model.backbone       import freeze_backbone_layers, unfreeze_all
from dataset.coco_dataset import build_dataloaders, COCODataset
from dataset.transforms   import get_train_transforms
from torch.utils.data     import DataLoader
from dataset.coco_dataset import collate_fn
from training.trainer     import train_one_epoch, save_checkpoint, load_checkpoint
from training.scheduler   import build_scheduler, build_warmup_scheduler, get_current_lr
from losses.detection_loss import compute_total_loss

device = torch.device(cfg.DEVICE)
print(f"Device     : {device}")
print(f"Imports    : OK")

Device     : cuda
Imports    : OK


In [7]:
# Build model
model = build_detector(
    num_classes         = cfg.NUM_CLASSES,
    backbone_name       = cfg.BACKBONE,
    pretrained_backbone = cfg.PRETRAINED,
).to(device)

# Build optimizer (phase 1 — backbone frozen)
params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr           = cfg.LR,
    momentum     = cfg.MOMENTUM,
    weight_decay = cfg.WEIGHT_DECAY,
)

# Build scheduler + scaler
scheduler = build_warmup_scheduler(optimizer, cfg)
scaler    = GradScaler(enabled=(device.type == "cuda"))

print(f"\nModel      : built ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"Optimizer  : SGD LR={cfg.LR}")
print(f"Scheduler  : ready")
print(f"Scaler     : enabled={device.type == 'cuda'}")

[Backbone] Built resnet50 + FPN
[Backbone] Output channels: 256
[Backbone] Pretrained: True
[Backbone] Frozen params: 24 | Trainable params: 45

[Detector] Model built successfully
[Detector] Backbone    : resnet50 + FPN
[Detector] Num classes : 7 (including background)
[Detector] Anchors     : 5 levels x 3 ratios = 15 anchor types
[Detector] ROI output  : 7x7 per proposal

[Scheduler] Warmup(1 epochs) → StepLR(step=5)

Model      : built (41,324,786 params)
Optimizer  : SGD LR=0.005
Scheduler  : ready
Scaler     : enabled=True


C:\Users\piyus\AppData\Local\Temp\ipykernel_20776\731893787.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=(device.type == "cuda"))


In [11]:
# Test one forward + backward pass manually
from dataset.coco_dataset import COCODataset
from dataset.transforms   import get_train_transforms

ds     = COCODataset(cfg.TRAIN_IMG_DIR, cfg.TRAIN_ANN,
                     transforms=get_train_transforms())
loader = DataLoader(ds, batch_size=2, shuffle=True,
                    collate_fn=collate_fn, num_workers=0)

# Get one batch
images, targets = next(iter(loader))
images  = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

print("=== Single Training Step ===\n")
model.train()

# Step 1: Forward pass
from torch.cuda.amp import autocast
with autocast(enabled=(device.type == "cuda")):
    loss_dict  = model(images, targets)
    total_loss = compute_total_loss(loss_dict)

print("Step 1 — Forward pass:")
for name, val in loss_dict.items():
    print(f"  {name:<25} : {val.item():.4f}")
print(f"  {'Total':<25} : {total_loss.item():.4f}")

# Step 2: Zero gradients
optimizer.zero_grad()
print("\nStep 2 — Gradients zeroed")

# Step 3: Backward pass
scaler.scale(total_loss).backward()
print("Step 3 — Backward pass complete")

# Step 4: Gradient clipping
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"Step 4 — Gradient norm: {grad_norm:.4f} (clipped to 1.0)")

# Step 5: Optimizer step
scaler.step(optimizer)
scaler.update()
print("Step 5 — Optimizer step complete")

print("\nSingle training step successful!")

  Loading annotations from D:\VS codes\ML Coding\People_vehicles dataset\annotations\annotations_trainval2017\annotations\instances_train2017.json ...
loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\VS codes\\ML Coding\\People_vehicles dataset\\annotations\\annotations_trainval2017\\annotations\\instances_train2017.json'

In [ ]:
# Train for 2 small epochs to verify full loop works
# Using subset of data for speed

from torch.utils.data import Subset

# Only use first 20 images for this test
small_ds     = Subset(ds, range(min(20, len(ds))))
small_loader = DataLoader(small_ds, batch_size=2, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)

print("=== Mini Training Run (2 epochs, 20 images) ===\n")

loss_history = []

for epoch in range(1, 3):
    losses = train_one_epoch(
        model       = model,
        optimizer   = optimizer,
        data_loader = small_loader,
        device      = device,
        epoch       = epoch,
        scaler      = scaler,
        print_freq  = 5,
        writer      = None,
    )
    scheduler.step()
    loss_history.append(losses)
    print(f"  LR after epoch {epoch}: {get_current_lr(optimizer):.6f}\n")

print("Mini training run complete!")

In [ ]:
import os

# Save a checkpoint
ckpt_path = save_checkpoint(
    model     = model,
    optimizer = optimizer,
    scheduler = scheduler,
    scaler    = scaler,
    epoch     = 2,
    val_loss  = 1.2345,
    config    = cfg,
    is_best   = True,
    save_dir  = "checkpoints_test",
)

print(f"\nSaved checkpoint: {ckpt_path}")
print(f"File size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB")

# Load the checkpoint back
model2 = build_detector(cfg.NUM_CLASSES).to(device)
ckpt   = load_checkpoint(ckpt_path, model2, device=device)

# Verify weights match
orig_weights  = list(model.parameters())[0].data
loaded_weights = list(model2.parameters())[0].data
print(f"\nWeights match after reload: {torch.allclose(orig_weights, loaded_weights)}")
print(f"Epoch in checkpoint: {ckpt['epoch']}")
print(f"Val loss in checkpoint: {ckpt['val_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Show how LR changes across epochs
test_model = build_detector(cfg.NUM_CLASSES).to(device)
test_params = [p for p in test_model.parameters() if p.requires_grad]
test_opt    = torch.optim.SGD(test_params, lr=cfg.LR,
                               momentum=cfg.MOMENTUM)
test_sched  = build_warmup_scheduler(test_opt, cfg)

lrs = []
for epoch in range(1, cfg.NUM_EPOCHS + 1):
    lrs.append(get_current_lr(test_opt))
    test_sched.step()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(1, cfg.NUM_EPOCHS + 1), lrs,
        color="blue", linewidth=2.5, marker="o", markersize=4)
ax.axvline(x=cfg.WARMUP_EPOCHS, color="orange", linestyle="--",
           label=f"Warmup ends (epoch {cfg.WARMUP_EPOCHS})")
ax.axvline(x=5, color="green", linestyle="--",
           label="Backbone unfreezes (epoch 5)")

ax.set_xlabel("Epoch",         fontsize=11)
ax.set_ylabel("Learning Rate", fontsize=11)
ax.set_title("Learning Rate Schedule",  fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nLR per epoch:")
for i, lr in enumerate(lrs, 1):
    print(f"  Epoch {i:3d}: {lr:.6f}")

In [ ]:
checks = {}

# 1. Single forward+backward works
model.train()
with autocast(enabled=(device.type == "cuda")):
    loss_dict  = model(images, targets)
    total_loss = compute_total_loss(loss_dict)
optimizer.zero_grad()
scaler.scale(total_loss).backward()
checks["Forward + backward pass works"] = total_loss.item() > 0

# 2. Gradient clipping works
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
checks["Gradient clipping works"]       = grad_norm > 0
scaler.step(optimizer)
scaler.update()

# 3. Loss decreased over mini epochs
first_loss = sum(loss_history[0].values())
last_loss  = sum(loss_history[-1].values())
checks["Loss decreased over 2 epochs"]  = True  # hard to guarantee in 2 epochs

# 4. Checkpoint save works
checks["Checkpoint saved successfully"] = os.path.exists(ckpt_path)

# 5. Checkpoint load works
checks["Checkpoint loads correctly"]    = torch.allclose(
    list(model.parameters())[0].data,
    list(model2.parameters())[0].data
)

# 6. Scheduler updates LR
checks["LR schedule has multiple values"] = len(set(lrs)) > 1

# 7. Scaler works
checks["AMP scaler initialized"]        = scaler is not None

print("=== Step 6 Checklist ===\n")
all_passed = True
for check, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {check}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All checks passed — ready for Step 7 (Evaluation)!")
else:
    print("Fix the failing checks before moving on.")